In [0]:
import requests
from requests.auth import HTTPBasicAuth
import pandas as pd
import xml.etree.ElementTree as ET
import urllib3
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    NullType, StringType, DateType,
    DecimalType, IntegerType, LongType, TimestampType
)
from pyspark.sql.functions import col, to_date, current_timestamp, lit, expr

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# ── Config ─────────────────────────────────────────────────────────────────
URL      = "https://poc-sapdevene.enesis.com:8020/sap/opu/odata/sap/ZCDS_LIPS_ODATA_CDS/ZCDS_LIPS_ODATA"
USERNAME = dbutils.secrets.get(scope="sap-odata-username", key="client_secret")
PASSWORD = dbutils.secrets.get(scope="sap-odata-password", key="client_secret")
HEADERS  = {"Accept": "application/atom+xml", "sap-client": "120"}
NS       = {
    "atom": "http://www.w3.org/2005/Atom",
    "m"   : "http://schemas.microsoft.com/ado/2007/08/dataservices/metadata",
    "d"   : "http://schemas.microsoft.com/ado/2007/08/dataservices"
}

lips_schema = {
    "Vbeln"                          : StringType(),
    "Posnr"                          : StringType(),        # NUMC
    "Pstyv"                          : StringType(),
    "Ernam"                          : StringType(),
    "Erzet"                          : StringType(),        # TIMS
    "Erdat"                          : DateType(),          # DATS — Created On
    "Matnr"                          : StringType(),
    "Matwa"                          : StringType(),
    "Matkl"                          : StringType(),
    "Werks"                          : StringType(),
    "Lgort"                          : StringType(),
    "Charg"                          : StringType(),
    "Lichn"                          : StringType(),
    "Kdmat"                          : StringType(),
    "Prodh"                          : StringType(),
    "Lfimg"                          : DecimalType(15, 3),  # QUAN — Actual Delivery Qty
    "Meins"                          : StringType(),        # UNIT
    "Vrkme"                          : StringType(),        # UNIT
    "Umvkz"                          : StringType(),        # DEC — keep string
    "Umvkn"                          : StringType(),        # DEC — keep string
    "Ntgew"                          : DecimalType(15, 3),  # QUAN — Net Weight
    "Brgew"                          : DecimalType(15, 3),  # QUAN — Gross Weight
    "Gewei"                          : StringType(),        # UNIT
    "Volum"                          : DecimalType(15, 3),  # QUAN — Volume
    "Voleh"                          : StringType(),        # UNIT
    "Kztlf"                          : StringType(),
    "Uebtk"                          : StringType(),
    "Uebto"                          : StringType(),        # DEC — keep string
    "Untto"                          : StringType(),        # DEC — keep string
    "Chspl"                          : StringType(),
    "Faksp"                          : StringType(),
    "Mbdat"                          : DateType(),          # DATS
    "Lgmng"                          : DecimalType(15, 3),  # QUAN — Storage Location Qty
    "Arktx"                          : StringType(),
    "Lgpbe"                          : StringType(),
    "Vbelv"                          : StringType(),
    "Posnv"                          : StringType(),        # NUMC
    "Vbtyv"                          : StringType(),
    "Vgsys"                          : StringType(),
    "Vgbel"                          : StringType(),
    "Vgpos"                          : StringType(),        # NUMC
    "Upflu"                          : StringType(),
    "Uepos"                          : StringType(),        # NUMC
    "Fkrel"                          : StringType(),
    "Ladgr"                          : StringType(),
    "Tragr"                          : StringType(),
    "Komkz"                          : StringType(),
    "Lgnum"                          : StringType(),
    "Lispl"                          : StringType(),
    "Lgtyp"                          : StringType(),
    "Lgpla"                          : StringType(),
    "Bwtex"                          : StringType(),
    "Bwart"                          : StringType(),
    "Bwlvs"                          : StringType(),        # NUMC
    "Kzdlg"                          : StringType(),
    "Bdart"                          : StringType(),
    "Plart"                          : StringType(),
    "Mtart"                          : StringType(),
    "Xchpf"                          : StringType(),
    "Xchar"                          : StringType(),
    "Vgref"                          : StringType(),
    "Posar"                          : StringType(),
    "Bwtar"                          : StringType(),
    "Sumbd"                          : StringType(),
    "Mtvfp"                          : StringType(),
    "Eannr"                          : StringType(),
    "Gsber"                          : StringType(),
    "Vkbur"                          : StringType(),
    "Vkgrp"                          : StringType(),
    "Vtweg"                          : StringType(),
    "Spart"                          : StringType(),
    "Grkor"                          : StringType(),        # NUMC
    "Fmeng"                          : StringType(),
    "Antlf"                          : StringType(),        # DEC — keep string
    "Vbeaf"                          : StringType(),        # DEC — keep string
    "Vbeav"                          : StringType(),        # DEC — keep string
    "Stafo"                          : StringType(),
    "Wavwr"                          : DecimalType(23, 2),  # CURR
    "Kzwi1"                          : DecimalType(23, 2),  # CURR
    "Kzwi2"                          : DecimalType(23, 2),  # CURR
    "Kzwi3"                          : DecimalType(23, 2),  # CURR
    "Kzwi4"                          : DecimalType(23, 2),  # CURR
    "Kzwi5"                          : DecimalType(23, 2),  # CURR
    "Kzwi6"                          : DecimalType(23, 2),  # CURR
    "Sobkz"                          : StringType(),
    "Aedat"                          : DateType(),          # DATS — Last Changed Date
    "Ean11"                          : StringType(),
    "Kvgr1"                          : StringType(),
    "Kvgr2"                          : StringType(),
    "Kvgr3"                          : StringType(),
    "Kvgr4"                          : StringType(),
    "Kvgr5"                          : StringType(),
    "Mvgr1"                          : StringType(),
    "Mvgr2"                          : StringType(),
    "Mvgr3"                          : StringType(),
    "Mvgr4"                          : StringType(),
    "Mvgr5"                          : StringType(),
    "Vpzuo"                          : StringType(),
    "Vgtyp"                          : StringType(),
    "Rfvgtyp"                        : StringType(),
    "Kostl"                          : StringType(),
    "Kokrs"                          : StringType(),
    "Paobjnr"                        : StringType(),
    "Prctr"                          : StringType(),
    "PsPspPnr"                       : StringType(),        # NUMC
    "Aufnr"                          : StringType(),
    "PosnrPp"                        : StringType(),        # NUMC
    "Kdauf"                          : StringType(),
    "Kdpos"                          : StringType(),        # NUMC
    "Vpmat"                          : StringType(),
    "Vpwrk"                          : StringType(),
    "Prbme"                          : StringType(),        # UNIT
    "Umref"                          : DecimalType(15, 6),  # FLTP
    "Knttp"                          : StringType(),
    "Kzvbr"                          : StringType(),
    "Fipos"                          : StringType(),
    "Fistl"                          : StringType(),
    "Geber"                          : StringType(),
    "Pckpf"                          : StringType(),
    "BedarLf"                        : StringType(),
    "Cmpnt"                          : StringType(),
    "Kcmeng"                         : DecimalType(15, 3),  # QUAN
    "Kcbrgew"                        : DecimalType(15, 3),  # QUAN
    "Kcntgew"                        : DecimalType(15, 3),  # QUAN
    "Kcvolum"                        : DecimalType(15, 3),  # QUAN
    "Uecha"                          : StringType(),        # NUMC
    "Cuobj"                          : StringType(),        # NUMC
    "CuobjCh"                        : StringType(),        # NUMC
    "Anzsn"                          : IntegerType(),       # INT4
    "Serail"                         : StringType(),
    "Kcgewei"                        : StringType(),        # UNIT
    "Kcvoleh"                        : StringType(),        # UNIT
    "Sernr"                          : StringType(),
    "Abrli"                          : StringType(),        # NUMC
    "Abart"                          : StringType(),
    "Abrvw"                          : StringType(),
    "Qplos"                          : StringType(),        # NUMC
    "Qtlos"                          : StringType(),        # NUMC
    "Nachl"                          : StringType(),
    "Magrv"                          : StringType(),
    "Objko"                          : StringType(),
    "Objpo"                          : StringType(),
    "Aeskd"                          : StringType(),
    "Shkzg"                          : StringType(),
    "Prosa"                          : StringType(),
    "Uepvw"                          : StringType(),
    "Empst"                          : StringType(),
    "Abtnr"                          : StringType(),
    "Koqui"                          : StringType(),
    "Stadat"                         : DateType(),          # DATS
    "Aktnr"                          : StringType(),
    "KnumhCh"                        : StringType(),
    "Prefe"                          : StringType(),
    "Exart"                          : StringType(),
    "Clint"                          : StringType(),        # NUMC
    "Chmvs"                          : StringType(),        # NUMC
    "Abeln"                          : StringType(),
    "Abelp"                          : StringType(),        # NUMC
    "LfimgFlo"                       : DecimalType(15, 6),  # FLTP
    "LgmngFlo"                       : DecimalType(15, 6),  # FLTP
    "KcmengFlo"                      : DecimalType(15, 6),  # FLTP
    "Kzumw"                          : StringType(),
    "Kmpmg"                          : DecimalType(15, 3),  # QUAN
    "Aurel"                          : StringType(),
    "Kpein"                          : StringType(),        # DEC — keep string
    "Kmein"                          : StringType(),        # UNIT
    "Netpr"                          : DecimalType(23, 2),  # CURR
    "Netwr"                          : DecimalType(23, 2),  # CURR
    "Kowrr"                          : StringType(),
    "Kzbew"                          : StringType(),
    "Mfrgr"                          : StringType(),
    "Chhpv"                          : StringType(),
    "Abfor"                          : StringType(),
    "Abges"                          : DecimalType(15, 6),  # FLTP
    "Mbuhr"                          : StringType(),        # TIMS
    "Wktnr"                          : StringType(),
    "Wktps"                          : StringType(),        # NUMC
    "J1bcfop"                        : StringType(),
    "J1btaxlw1"                      : StringType(),
    "J1btaxlw2"                      : StringType(),
    "J1btxsdc"                       : StringType(),
    "Situa"                          : StringType(),
    "Rsnum"                          : StringType(),        # NUMC
    "Rspos"                          : StringType(),        # NUMC
    "Rsart"                          : StringType(),
    "Kannr"                          : StringType(),
    "Kzfme"                          : StringType(),
    "Profl"                          : StringType(),
    "Kcmengvme"                      : DecimalType(15, 3),  # QUAN
    "Kcmengvmef"                     : DecimalType(15, 6),  # FLTP
    "Kzbws"                          : StringType(),
    "Pspnr"                          : StringType(),        # NUMC
    "Eprio"                          : StringType(),
    "Rules"                          : StringType(),
    "Kzbef"                          : StringType(),
    "Mprof"                          : StringType(),
    "Ematn"                          : StringType(),
    "Lgbzo"                          : StringType(),
    "Handle"                         : StringType(),
    "Verurpos"                       : StringType(),        # NUMC
    "Lifexpos"                       : StringType(),        # NUMC
    "Noatp"                          : StringType(),
    "Nopck"                          : StringType(),
    "Rblvs"                          : StringType(),        # NUMC
    "Berid"                          : StringType(),
    "Bestq"                          : StringType(),
    "Umbsq"                          : StringType(),
    "Ummat"                          : StringType(),
    "Umwrk"                          : StringType(),
    "Umlgo"                          : StringType(),
    "Umcha"                          : StringType(),
    "Umbar"                          : StringType(),
    "Umsok"                          : StringType(),
    "Sonum"                          : StringType(),
    "Usonu"                          : StringType(),
    "Akkur"                          : StringType(),        # DEC — keep string
    "Akmng"                          : StringType(),
    "Vkgru"                          : StringType(),
    "ShkzgUm"                        : StringType(),
    "Insmk"                          : StringType(),
    "Kzech"                          : StringType(),
    "Flgwm"                          : StringType(),
    "Berkz"                          : StringType(),
    "Hupos"                          : StringType(),
    "Nowab"                          : StringType(),
    "Konto"                          : StringType(),
    "Kzear"                          : StringType(),
    "Hsdat"                          : DateType(),          # DATS
    "Vfdat"                          : DateType(),          # DATS — Shelf Life Expiration Date
    "Lfgja"                          : StringType(),        # NUMC
    "Lfbnr"                          : StringType(),
    "Lfpos"                          : StringType(),        # NUMC
    "Grund"                          : StringType(),        # NUMC
    "Fobwa"                          : StringType(),
    "Dlvtp"                          : StringType(),
    "Exbwr"                          : DecimalType(23, 2),  # CURR
    "Bpmng"                          : DecimalType(15, 3),  # QUAN
    "Exvkw"                          : DecimalType(23, 2),  # CURR
    "CmpreFlt"                       : DecimalType(15, 6),  # FLTP
    "Kzpod"                          : StringType(),
    "Lfdez"                          : StringType(),
    "Umrev"                          : DecimalType(15, 6),  # FLTP
    "Podrel"                         : StringType(),
    "Kzuml"                          : StringType(),
    "Fkber"                          : StringType(),
    "GrantNbr"                       : StringType(),
    "Kzwso"                          : StringType(),
    "Gmcontrol"                      : StringType(),
    "PostingChange"                  : StringType(),
    "UmPsPspPnr"                     : StringType(),        # NUMC
    "PreVlEtens"                     : StringType(),        # NUMC
    "SpeGenElikz"                    : StringType(),
    "SpeScrapInd"                    : StringType(),
    "SpeAuthNumber"                  : StringType(),
    "SpeInspoutGuid"                 : StringType(),        # RAW — keep string
    "SpeFollowUp"                    : StringType(),
    "SpeExpDateExt"                  : StringType(),        # DEC — timestamp numerik SAP
    "SpeExpDateInt"                  : StringType(),        # DEC — timestamp numerik SAP
    "SpeAuthComplet"                 : StringType(),
    "Ormng"                          : DecimalType(15, 3),  # QUAN
    "SpeAtpTmstmp"                   : StringType(),        # DEC — timestamp numerik SAP
    "SpeOrigSys"                     : StringType(),
    "SpeLieffz"                      : DecimalType(15, 3),  # QUAN
    "SpeImwrk"                       : StringType(),
    "SpeLifexpos2"                   : StringType(),
    "SpeExceptCode"                  : StringType(),
    "SpeKeepQty"                     : DecimalType(15, 3),  # QUAN
    "SpeAlternate"                   : StringType(),
    "SpeMatSubst"                    : StringType(),
    "SpeStruc"                       : StringType(),        # NUMC
    "SpeApoQntyfac"                  : StringType(),        # DEC — keep string
    "SpeApoQntydiv"                  : StringType(),        # DEC — keep string
    "SpeHerkl"                       : StringType(),
    "SpeBxpDateExt"                  : StringType(),        # DEC — timestamp numerik SAP
    "SpeVersion"                     : StringType(),        # NUMC
    "SpeComplMvt"                    : StringType(),
    "J1btaxlw4"                      : StringType(),
    "J1btaxlw5"                      : StringType(),
    "J1btaxlw3"                      : StringType(),
    "BudgetPd"                       : StringType(),
    "Kbnkz"                          : StringType(),
    "FarrReltype"                    : StringType(),
    "Sitkz"                          : StringType(),
    "SgtRcat"                        : StringType(),
    "SgtScat"                        : StringType(),
    "Besta"                          : StringType(),
    "Cmppi"                          : StringType(),
    "Cmppj"                          : StringType(),
    "Fkivp"                          : StringType(),
    "Fksta"                          : StringType(),
    "Gbsta"                          : StringType(),
    "Hdall"                          : StringType(),
    "Koqua"                          : StringType(),
    "Kosta"                          : StringType(),
    "Lvsta"                          : StringType(),
    "Pdsta"                          : StringType(),
    "Pksta"                          : StringType(),
    "Uvall"                          : StringType(),
    "Uvfak"                          : StringType(),
    "Uvpak"                          : StringType(),
    "Uvpik"                          : StringType(),
    "Uvvlk"                          : StringType(),
    "Uvwak"                          : StringType(),
    "Vlstp"                          : StringType(),
    "Wbsta"                          : StringType(),
    "Uvp01"                          : StringType(),
    "Uvp02"                          : StringType(),
    "Uvp03"                          : StringType(),
    "Uvp04"                          : StringType(),
    "Uvp05"                          : StringType(),
    "Emcst"                          : StringType(),
    "Slcst"                          : StringType(),
    "Lccst"                          : StringType(),
    "xsapmpxlbasp"                   : StringType(),
    "Dataaging"                      : DateType(),          # DATS
    "xcwmxlfimg"                     : DecimalType(15, 3),  # QUAN
    "xcwmxlfime"                     : StringType(),        # UNIT
    "xcwmxpikmg"                     : DecimalType(15, 3),  # QUAN
    "xcwmxpikme"                     : StringType(),        # UNIT
    "xcwmxxenter"                    : StringType(),
    "xcwmxxtaenter"                  : StringType(),
    "xcwmxkcmeng"                    : DecimalType(15, 3),  # QUAN
    "xcwmxebumg"                     : DecimalType(15, 3),  # QUAN
    "DummyDelitmInclEewPs"           : StringType(),
    "xkjedmxinverted"                : StringType(),
    "xsapmpxlbaNo"                   : StringType(),
    "xsapmpxaltConv"                 : StringType(),
    "Lgtor"                          : StringType(),
    "FshSeasonYear"                  : StringType(),
    "FshSeason"                      : StringType(),
    "FshCollection"                  : StringType(),
    "FshTheme"                       : StringType(),
    "FshKvgr6"                       : StringType(),
    "FshKvgr7"                       : StringType(),
    "FshKvgr8"                       : StringType(),
    "FshKvgr9"                       : StringType(),
    "FshKvgr10"                      : StringType(),
    "FshVasRel"                      : StringType(),
    "FshVasPrntId"                   : StringType(),        # NUMC
    "FshTransaction"                 : StringType(),
    "FshItemGroup"                   : StringType(),        # NUMC
    "FshItem"                        : StringType(),        # NUMC
    "FshRsnum"                       : StringType(),        # NUMC
    "FshRspos"                       : StringType(),        # NUMC
    "MillUcdet"                      : StringType(),
    "ConsOrder"                      : StringType(),
    "WrfCharstc1"                    : StringType(),
    "WrfCharstc2"                    : StringType(),
    "WrfCharstc3"                    : StringType(),
}


spark = SparkSession.builder.getOrCreate()


# ── Main function ──────────────────────────────────────────────────────────
def ingest_lips_bronze(start_date: str, end_date: str, write_mode: str = "overwrite"):
    """
    Fetch LIPS dari SAP OData dan simpan ke Bronze Delta table.

    Parameters
    ----------
    start_date : str  — format 'YYYY-MM-DD', contoh '2019-09-01'
    end_date   : str  — format 'YYYY-MM-DD', contoh '2019-09-30'
    write_mode : str  — 'overwrite' atau 'append' (default: 'overwrite')
    """

    print(f"\n{'='*60}")
    print(f"📦 Fetching LIPS: {start_date} → {end_date}")
    print(f"{'='*60}")

    # ── Fetch dengan pagination ────────────────────────────────
    records = []
    page    = 0

    while True:
        params = {
            "$filter": f"Erdat ge datetime'{start_date}T00:00:00' and Erdat le datetime'{end_date}T23:59:59'",
        }

        response = requests.get(
            URL,
            auth    = HTTPBasicAuth(USERNAME, PASSWORD),
            headers = HEADERS,
            params  = params,
            verify  = False,
            timeout = 300  # naikkan timeout karena data bisa besar
        )
        print(f"  status={response.status_code} | size={len(response.content)} bytes")

        try:
            root = ET.fromstring(response.content)
        except ET.ParseError as e:
            raise RuntimeError(f"Failed to parse XML: {e}")

        root_tag = root.tag.split("}")[-1]
        batch    = []

        if root_tag == "feed":
            for entry in root.findall("atom:entry", NS):
                props = entry.find(".//m:properties", NS)
                if props is not None:
                    batch.append({child.tag.split("}")[-1]: child.text for child in props})
        elif root_tag == "entry":
            props = root.find(".//m:properties", NS)
            if props is not None:
                batch.append({child.tag.split("}")[-1]: child.text for child in props})

        records.extend(batch)
        print(f"  → Fetched: {len(batch):,} rows | Total: {len(records):,}")
        break  # SAP return semua sekaligus, tidak perlu loop
            

    # ── pandas → Spark ─────────────────────────────────────────
    df       = pd.DataFrame(records)
    spark_df = spark.createDataFrame(df)

    # ── Cast schema ────────────────────────────────────────────
    for col_name, col_type in lips_schema.items():
        if col_name in spark_df.columns:
            if isinstance(col_type, DateType):
                spark_df = spark_df.withColumn(
                    col_name,
                    expr(f"try_to_date(`{col_name}`, 'yyyy-MM-dd\\'T\\'HH:mm:ss')")
                )
            else:
                spark_df = spark_df.withColumn(col_name, col(col_name).cast(col_type))

    # ── Fix NullType ───────────────────────────────────────────
    for field in spark_df.schema.fields:
        if isinstance(field.dataType, NullType):
            spark_df = spark_df.withColumn(field.name, spark_df[field.name].cast(StringType()))

    # ── Audit columns ──────────────────────────────────────────
    spark_df = spark_df \
        .withColumn("_ingestion_timestamp", current_timestamp()) \
        .withColumn("_source", lit("ZCDS_LIPS_ODATA"))

    # ── Write ──────────────────────────────────────────────────
    spark_df.write \
        .format("delta") \
        .mode(write_mode) \
        .option("overwriteSchema", "true") \
        .saveAsTable("poc_enesis.bronze.ZCDS_LIPS_ODATA")

    print(f"  ✅ Written to Bronze ({write_mode})")
    spark.sql("SELECT COUNT(*) as total FROM poc_enesis.bronze.ZCDS_LIPS_ODATA").show()

In [0]:
print("========= BATCH 1: Januari - September (part 1) =========")
ingest_lips_bronze("2019-01-01", "2019-09-14", write_mode="overwrite") 

In [0]:
print("========= BATCH 2: September (part 2)=========")
ingest_lips_bronze("2019-09-15", "2019-09-30", write_mode="append") 

In [0]:
print("========= BATCH 3: Oktober (part 1) =========")
ingest_lips_bronze("2019-10-01", "2019-10-14", write_mode="append") 

In [0]:
print("========= BATCH 4: Oktober (part 2) =========")
ingest_lips_bronze("2019-10-15", "2019-10-31", write_mode="append") 

In [0]:
print("========= BATCH 5: November (part 1) =========")
ingest_lips_bronze("2019-11-01", "2019-11-14", write_mode="append") 

In [0]:
print("========= BATCH 6: November (part 2) =========")
ingest_lips_bronze("2019-11-15", "2019-11-30", write_mode="append") 

In [0]:
print("========= BATCH 7: Desember (part 1) =========")
ingest_lips_bronze("2019-12-01", "2019-12-14", write_mode="append") 

In [0]:
print("========= BATCH 8: Desember (part 2) =========")
ingest_lips_bronze("2019-12-15", "2019-12-31", write_mode="append") 

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp, lit, to_date

watermark_row = spark.createDataFrame([Row(
    source_name   = "SAP_ODATA",
    table_name    = "ZCDS_LIPS_ODATA",
    last_run_date = RUN_DATE[:10],
    last_run_ts   = None,
    status        = "SUCCESS",
    rows_ingested = int(spark_df.count())
)])

watermark_row = watermark_row \
    .withColumn("last_run_date", to_date("last_run_date")) \
    .withColumn("last_run_ts", current_timestamp())

from delta.tables import DeltaTable
wm = DeltaTable.forName(spark, "poc_enesis.bronze.pipeline_watermark")
wm.alias('t').merge(
    watermark_row.alias('s'),
    't.source_name = s.source_name AND t.table_name = s.table_name'
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
print("✅ Watermark updated")


In [0]:
import uuid
from datetime import datetime

run_id    = str(uuid.uuid4())[:8]
started   = datetime.utcnow()
SCHEMA     = 'BRONZE'
TABLE     = 'ZCDS_LIPS_ODATA'

try:
    # ... semua logic ingest & write ke bronze ...
    rows_in = spark_df.count()
    status  = 'SUCCESS'
    msg     = f'{rows_in} rows ingested'
except Exception as e:
    rows_in = 0
    status  = 'FAILED'
    msg     = str(e)[:500]
    raise  # re-raise supaya Job tau pipeline gagal
finally:
    ended = datetime.utcnow()
    log_df = spark.createDataFrame([{
        'run_id': run_id, 'schema_name': SCHEMA, 'table_name': TABLE,
        'status': status, 'message': msg, 'rows_in': rows_in, 'rows_out': 0,
        'started_at': started, 'ended_at': ended
    }])
    log_df.write.format('delta').mode('append').saveAsTable(
        'poc_enesis.bronze.run_log'
    )
